## Purpose
This notebook takes a list of (often historical) **NDC package codes** and tries to infer **currently listed** FDA package NDC(s) that appear to belong to the *same product*.

**Important**: this is **not** a true historical→current mapping table. It is a **heuristic lookup** that:
- normalizes each input to **11 digits**
- guesses a small set of plausible `product_ndc` formats
- queries openFDA `drug/ndc` for a matching product
- returns **all current package NDCs** listed under that product (when found)

## Why lookups fail (and why 404 is expected)
openFDA commonly returns **HTTP 404** when a query has **0 matches**. That does *not* necessarily mean the drug never existed.

Common reasons:
- **Historically existed but no longer listed** (discontinued / relabeled / obsolete packages)
- **openFDA is not a complete historical archive** (especially older or non‑SPL era products)
- **Segment-structure guess is wrong** (the 11-digit code may not correspond to the assumed labeler-product-package split)

This notebook keeps those “no match” rows so you can audit what was not found.

## Inputs
- `Asthma NDCs_5.7.2026.xlsx`
  - `NDC`: package-level NDC in any common representation (digits, hyphens, etc.)
  - `asthma_med_class_comp`: used to exclude rows marked `Non_asthma_med`

## Output(s)
Batch Excel files (to avoid very large single writes):
- `historical_to_current_ndc_mapping_batch_001.xlsx`, `..._002.xlsx`, ...

Each output row represents either:
- **No match** for the historical NDC (one row, mostly nulls), or
- **One matched current package NDC** (one row per package)

Columns:
- `historical_ndc_11`: normalized historical NDC (11 digits)
- `matched_product_ndc`: the `product_ndc` format that matched (when any)
- `generic_name`, `brand_name`: from openFDA product record
- `current_package_ndc`: current FDA package code (often hyphenated)
- `current_package_ndc_11`: normalized 11-digit version of `current_package_ndc`
- `exists_in_original_file`: whether `current_package_ndc_11` was present in the original spreadsheet

## High-level workflow
- Load spreadsheet → filter to asthma meds → normalize NDCs to 11 digits
- For each unique NDC (in batches):
  - try a few `product_ndc` variants
  - if found, emit one row per current package NDC
  - if not found, emit a single “no match” row

## Notes / caveats
- openFDA results can change over time (products added/removed).
- This workflow maximizes **recall** for “what looks related”, not perfect longitudinal truth.
- If you need historical label recall, use `drug/label` as an additional endpoint (example code is provided later as a commented-out option).
- Finally, **clinician's manual review** is necessary to confirm that the identified NDC codes are truly asthma-related.


In [2]:
import os
import re
import time
from typing import Iterable, Iterator, Optional

import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry

# =========================================================
# How to read this cell (what it does)
# =========================================================
# Goal:
# - Starting from historical NDCs (package-level identifiers), enumerate candidate
#   *currently listed* FDA package NDCs via openFDA `drug/ndc`.
#
# Key ideas in this notebook:
# - openFDA `drug/ndc` is primarily searched by `product_ndc` (labeler-product),
#   not by full package NDC. Therefore we generate several plausible `product_ndc`
#   variants from the 11-digit normalized code and try them.
# - openFDA often returns HTTP 404 to represent "0 matches" for a search.
#   In this workflow, 404 is treated as an expected "no hit" outcome.
# - One `product_ndc` query can return multiple product records, and each product
#   can include multiple `package_ndc` values. We keep *all* candidates to support
#   downstream auditing and clinical review.
#
# Execution design:
# - Restartable batch outputs: if `*_batch_###.xlsx` already exists, that batch is skipped.

# =========================================================
# Step 0) Config (inputs / column names / outputs / runtime parameters)
# =========================================================
# Input Excel (read with dtype=str so leading zeros are preserved).
INPUT_XLSX = "Asthma NDCs_5.7.2026.xlsx"

# Expected column names in INPUT_XLSX.
# - `NDC_COLUMN` may contain digits-only, hyphenated, 10/11-digit, or Excel-coerced values.
# - `CLASS_COLUMN` is used to exclude rows labeled as Non_asthma_med (upstream curation).
NDC_COLUMN = "NDC"
CLASS_COLUMN = "asthma_med_class_comp"

# Output prefix for batch files.
OUTPUT_PREFIX = "historical_to_current_ndc_mapping"

# Batch size: larger = fewer files, smaller = easier restart granularity.
BATCH_SIZE = 500

# Delay between lookups (rate-limit mitigation).
SLEEP = 0.05

# =========================================================
# Step 1) HTTP session for openFDA (with retry/backoff)
# =========================================================
def get_session() -> requests.Session:
    """Create a `requests.Session` for openFDA with retry/backoff.

    Retry targets:
    - 429: Too Many Requests (may recover after waiting)
    - 5xx: transient server errors

    Not retried (treated as normal in this workflow):
    - 404: openFDA often uses 404 to mean "no search results"
    """

    session = requests.Session()

    retries = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )

    session.mount("https://", HTTPAdapter(max_retries=retries))
    return session


http = get_session()

# =========================================================
# Step 2) Normalize NDC → 11-digit digits-only
# =========================================================
def clean_ndc11(value: object) -> Optional[str]:
    """Normalize an input value to an 11-digit, digits-only NDC string.

    Why this is needed:
    - NDC representations are often inconsistent in spreadsheets/CSVs (hyphens, variable
      segment widths, 10 vs 11 digits, Excel auto-conversion to numeric, whitespace, etc.).
    - This workflow needs a single stable key for dedupe / comparisons / existence checks.

    Rules:
    - Convert to string → strip non-digits → if length is 1..11, left-pad with zeros to 11.
    - If >11 digits after stripping, do not guess; return None for auditing.
    """

    if value is None:
        return None

    s = str(value).strip()
    if s == "" or s.lower() == "nan":
        return None

    # Remove hyphens/spaces/any non-digit characters.
    digits = re.sub(r"\D", "", s)

    if 1 <= len(digits) <= 11:
        return digits.zfill(11)

    return None


# =========================================================
# Step 3) Generate `product_ndc` candidates (heuristic)
# =========================================================
def product_ndc_variants(ndc11: str) -> list[str]:
    """Generate several plausible `product_ndc` (labeler-product) strings.

    Background:
    - openFDA `drug/ndc` searches are typically keyed by `product_ndc` (e.g., 00004-2906),
      not by the full package code.
    - Because our input is a normalized 11-digit *package-level* identifier, the original
      labeler/product segmentation is ambiguous and depends on the original 10-digit format.

    We therefore try a small set of common interpretations to improve recall.
    """

    s = ndc11
    return [
        f"{s[0:5]}-{s[5:9]}",
        f"{s[1:5]}-{s[5:9]}",
        f"{s[0:5]}-{s[6:9]}",
    ]


# =========================================================
# Step 4) Query openFDA (return *all* candidates)
# =========================================================
def lookup_all_matches(ndc11: str) -> Optional[list[dict]]:
    """Return all openFDA `drug/ndc` hits for a given NDC (across all variants).

    Output row granularity:
    - Typically one row per (product result) × (package)
    - If `packaging` is missing/empty, keep a single product-level row (package fields null)

    Return semantics:
    - None: no hits for any `product_ndc` candidate
    - list[dict]: one or more hits (kept in full for auditing/review)
    """

    all_rows: list[dict] = []

    for product_ndc in product_ndc_variants(ndc11):
        url = (
            "https://api.fda.gov/drug/ndc.json"
            f'?search=product_ndc:"{product_ndc}"&limit=100'
        )

        try:
            r = http.get(url, timeout=10)

            # openFDA may return HTTP 404 to indicate "0 matches".
            # Treat that as a normal no-hit for this candidate.
            if r.status_code == 404:
                continue

            r.raise_for_status()

            results = r.json().get("results", [])
            if not results:
                continue

            # A single product_ndc query may return multiple product records; iterate them all.
            for item in results:
                generic = item.get("generic_name")
                brand = item.get("brand_name")
                packages = item.get("packaging", [])

                # If packaging is missing/empty, still preserve a product-level hit.
                if not packages:
                    all_rows.append(
                        {
                            "matched_product_ndc": product_ndc,
                            "generic_name": generic,
                            "brand_name": brand,
                            "package_ndc": None,
                            "package_ndc_11": None,
                        }
                    )
                    continue

                # Otherwise, explode to one row per package_ndc.
                for p in packages:
                    package_ndc = p.get("package_ndc")
                    all_rows.append(
                        {
                            "matched_product_ndc": product_ndc,
                            "generic_name": generic,
                            "brand_name": brand,
                            "package_ndc": package_ndc,
                            "package_ndc_11": clean_ndc11(package_ndc),
                        }
                    )

        except Exception as e:
            # Beyond 404, network and transient failures can occur; keep a log for auditing.
            print(f"NO HIT / ERROR: {product_ndc} -> {e}")

    return all_rows if all_rows else None


# =========================================================
# Step 5) Batching utility
# =========================================================
def chunked(iterable: Iterable[str], size: int = 500) -> Iterator[list[str]]:
    """Split an iterable into lists of length `size`."""

    items = list(iterable)
    for i in range(0, len(items), size):
        yield items[i : i + size]


# =========================================================
# Step 6) Load → filter → normalize → uniquify
# =========================================================
# NDCs require leading zeros; read as strings (dtype=str).
df = pd.read_excel(INPUT_XLSX, dtype=str)

# Exclude rows explicitly tagged as non-asthma.
# - This does not guarantee indication (asthma vs COPD); it only mirrors upstream curation.
df = df[df[CLASS_COLUMN].fillna("") != "Non_asthma_med"].copy()

# Normalize to a comparable key (11-digit digits-only).
df["NDC_11"] = df[NDC_COLUMN].apply(clean_ndc11)

# Drop rows we cannot parse.
# If you want an audit output, you can export the dropped rows before filtering.
df = df[df["NDC_11"].notna()].copy()

# Set of NDCs present in the original file (used for existence flags later).
original_ndc_set = set(df["NDC_11"])

# Worklist = unique NDC_11 values (one lookup per unique identifier).
sorted_ndcs = sorted(original_ndc_set)

print(f"Total unique asthma NDCs: {len(sorted_ndcs)}")


# =========================================================
# Step 7) Main loop (NDC → enumerate candidates → batch save)
# =========================================================
for batch_idx, batch in enumerate(chunked(sorted_ndcs, BATCH_SIZE), 1):
    output_file = f"{OUTPUT_PREFIX}_batch_{batch_idx:03d}.xlsx"

    # Restartable behavior: if output exists, treat the batch as complete and skip.
    if os.path.exists(output_file):
        print(f"Skip existing file: {output_file}")
        continue

    rows: list[dict] = []

    print("\n==============================")
    print(f"Batch {batch_idx} ({len(batch)} NDCs)")
    print("==============================")

    for i, ndc11 in enumerate(batch, 1):
        print(f"[Batch {batch_idx}] {i}/{len(batch)} : {ndc11}")

        hits = lookup_all_matches(ndc11)

        # No hits (could be historical-only / segmentation guess wrong / not present in openFDA, etc.)
        if hits is None:
            rows.append(
                {
                    "historical_ndc_11": ndc11,
                    "matched_product_ndc": None,
                    "generic_name": None,
                    "brand_name": None,
                    "current_package_ndc": None,
                    "current_package_ndc_11": None,
                    "exists_in_original_file": None,
                }
            )

        # One or more hits: keep all candidates as rows for downstream review.
        else:
            for h in hits:
                pkg = h["package_ndc"]
                pkg11 = clean_ndc11(pkg)

                rows.append(
                    {
                        "historical_ndc_11": ndc11,
                        "matched_product_ndc": h["matched_product_ndc"],
                        "generic_name": h["generic_name"],
                        "brand_name": h["brand_name"],
                        "current_package_ndc": pkg,
                        "current_package_ndc_11": pkg11,
                        # Whether the candidate package NDC already exists in the original list
                        # (useful as a clue for identical/duplicate detection).
                        "exists_in_original_file": (pkg11 in original_ndc_set) if pkg11 else None,
                    }
                )

        # Rate-limit mitigation.
        time.sleep(SLEEP)

    # Save this batch.
    out_df = pd.DataFrame(rows)
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        out_df.to_excel(writer, sheet_name="all_mappings", index=False)

    print(f"Saved: {output_file} ({len(out_df)} rows)")


Total unique asthma NDCs: 3851
Skip existing file: historical_to_current_ndc_mapping_batch_001.xlsx
Skip existing file: historical_to_current_ndc_mapping_batch_002.xlsx
Skip existing file: historical_to_current_ndc_mapping_batch_003.xlsx
Skip existing file: historical_to_current_ndc_mapping_batch_004.xlsx

Batch 5 (500 NDCs)
[Batch 5] 1/500 : 49137014001
[Batch 5] 2/500 : 49137014224
[Batch 5] 3/500 : 49260050219
[Batch 5] 4/500 : 49326027402
[Batch 5] 5/500 : 49326030504
[Batch 5] 6/500 : 49326034616
[Batch 5] 7/500 : 49326034790
[Batch 5] 8/500 : 49326037704
[Batch 5] 9/500 : 49326038890
[Batch 5] 10/500 : 49326041490
[Batch 5] 11/500 : 49326041804
[Batch 5] 12/500 : 49326042504
[Batch 5] 13/500 : 49326050390
[Batch 5] 14/500 : 49326051090
[Batch 5] 15/500 : 49326092904
[Batch 5] 16/500 : 49348083861
[Batch 5] 17/500 : 49452005301
[Batch 5] 18/500 : 49452005302
[Batch 5] 19/500 : 49452005303
[Batch 5] 20/500 : 49452022501
[Batch 5] 21/500 : 49452022502
[Batch 5] 22/500 : 49452022503


In [2]:
import glob
import pandas as pd

# ---------------------------------------------------------
# Input / Output
# ---------------------------------------------------------
INPUT_PATTERN = "historical_to_current_ndc_mapping_batch_*.xlsx"
OUTPUT_FILE = "historical_to_current_ndc_mapping_COMBINED.xlsx"

# ---------------------------------------------------------
# Step 1: Load all batch files
# ---------------------------------------------------------
files = sorted(glob.glob(INPUT_PATTERN))

print(f"Found {len(files)} files")

dfs = []

for f in files:
    print(f"Loading: {f}")
    df = pd.read_excel(f, sheet_name="all_mappings", dtype=str)
    dfs.append(df)

# ---------------------------------------------------------
# Step 2: Concatenate all
# ---------------------------------------------------------
all_df = pd.concat(dfs, ignore_index=True)

print(f"Total rows before filtering: {len(all_df)}")

# ---------------------------------------------------------
# Step 3: Keep only "hits"
# ---------------------------------------------------------
filtered_df = all_df[
    all_df["matched_product_ndc"].notna()
].copy()

print(f"After FDA match filter: {len(filtered_df)}")

# ---------------------------------------------------------
# Step 4: Keep only NEW NDCs (not in original file)
# ---------------------------------------------------------
filtered_df = filtered_df[
    filtered_df["exists_in_original_file"] == "False"
].copy()

print(f"After removing existing NDCs: {len(filtered_df)}")

# ---------------------------------------------------------
# Step 5: Save combined file
# ---------------------------------------------------------
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    filtered_df.to_excel(writer, sheet_name="combined_hits_new_only", index=False)

print(f"\nSaved: {OUTPUT_FILE}")

Found 8 files
Loading: historical_to_current_ndc_mapping_batch_001.xlsx
Loading: historical_to_current_ndc_mapping_batch_002.xlsx
Loading: historical_to_current_ndc_mapping_batch_003.xlsx
Loading: historical_to_current_ndc_mapping_batch_004.xlsx
Loading: historical_to_current_ndc_mapping_batch_005.xlsx
Loading: historical_to_current_ndc_mapping_batch_006.xlsx
Loading: historical_to_current_ndc_mapping_batch_007.xlsx
Loading: historical_to_current_ndc_mapping_batch_008.xlsx
Total rows before filtering: 5159
After FDA match filter: 2175
After removing existing NDCs: 1690

Saved: historical_to_current_ndc_mapping_COMBINED.xlsx
